# Notebook 17 visual report definitions
Purpose: render measured input preparation, kernel approximation, complete JEPA cost and paired quality. Inputs: saved source-bound JSON receipts; no fresh data. Outputs: portable HTML, PNG figures and inspectable result JSON. Runtime: seconds; no training on import. Checkpoints/resume: caller owns scientific checkpoints and immutable phase clocks. Undefined measurements remain UNKNOWN; software fixtures are labeled.

In [ ]:
from pathlib import Path
import base64, html, json, os, types, uuid
if "__file__"not in globals():__file__=str(Path(os.environ.get("FMM_JEPA_ROOT","/home/benjamin/workspace/ahr_go/fmm_jeppa"))/"experiments/17_fixed_kernel/report.ipynb")
ROOT=Path(__file__).resolve().parents[2]

def render(root,artifact,result,bench=None,cost=None,pipeline=None,screen=None):
    """Portable measured-evidence report; fixtures and incomplete cases are explicit."""
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    import numpy as np
    artifact=Path(artifact);artifact.mkdir(parents=True,exist_ok=True);figures=[]
    def saved_evidence(key,all_key=None):
        record=result.get(key)
        if record is None and all_key and result.get(all_key):record=result[all_key][-1]
        if not record:return None
        path=Path(record['path']).resolve()
        if not path.is_relative_to(Path(root).resolve()/'artifacts'):raise ValueError('Report evidence lies outside project artifacts')
        import hashlib
        if hashlib.sha256(path.read_bytes()).hexdigest()!=record['sha256']:raise ValueError('Report evidence checksum differs')
        return json.loads(path.read_text())
    if bench is None:bench=saved_evidence('benchmark')
    if cost is None:cost=saved_evidence('cost','all_cost_receipts')
    if pipeline is None:pipeline=saved_evidence('pipeline','all_pipeline_receipts')
    fixture=bool(result.get("fixture_source")or(bench or {}).get("profile")=="validation")
    prefix="Software fixture — "if fixture else""
    colors={32:"#687881",64:"#315f8c",128:"#c27228","sdpa":"#315f8c","fixed":"#c27228"}
    plt.rcParams.update({"font.size":11,"axes.spines.top":False,"axes.spines.right":False})
    def save(fig,name):
        fig.savefig(artifact/name,dpi=150,bbox_inches="tight");plt.close(fig);figures.append(name)
    if pipeline and pipeline.get("rows"):
        rows=pipeline["rows"];fig,ax=plt.subplots(figsize=(8,4),layout="constrained")
        for row in rows:
            worker=row["workers"];values=row["seconds"]
            ax.scatter([worker]*len(values),values,s=30,color=colors["sdpa"])
            ax.scatter([worker],[row["median_seconds"]],marker="D",s=65,color=colors["fixed"]if worker==pipeline["selected_workers"]else colors["sdpa"])
        ax.set_xticks([r["workers"]for r in rows]);ax.set_ylim(bottom=0);ax.set_xlabel("Ordered image-preparation workers");ax.set_ylabel("Preparation seconds per effective batch")
        ax.set_title(prefix+"Shared TRAIN-only input preparation; dots are repeats")
        ax.text(.98,.96,"Selected workers: "+str(pipeline["selected_workers"])+"; identical settings for both models",transform=ax.transAxes,va="top",ha="right",fontsize=9)
        save(fig,"input_pipeline.png")
    if bench and bench.get("rows"):
        rows=bench["rows"];counts=sorted({int(r["tokens"])for r in rows});ranks=sorted({int(r["rank"])for r in rows});fig,axes=plt.subplots(2,2,figsize=(12,8),layout="constrained")
        def worst(rank,key):
            values=[]
            for count in counts:
                group=[r for r in rows if int(r["rank"])==rank and int(r["tokens"])==count]
                if not group or any(r.get("status")!="MEASURED"for r in group):values.append(np.nan);continue
                if key=="time":raw=[r["measurements"]["fixed"]["forward_backward_seconds"]/r["measurements"]["sdpa"]["forward_backward_seconds"]for r in group]
                elif key=="memory":raw=[None if r["measurements"]["fixed"].get("peak_cuda_allocated_bytes")is None or r["measurements"]["sdpa"].get("peak_cuda_allocated_bytes")is None else r["measurements"]["fixed"]["peak_cuda_allocated_bytes"]/r["measurements"]["sdpa"]["peak_cuda_allocated_bytes"]for r in group]
                else:raw=[r["errors"][key]for r in group]
                values.append(max(raw)if all(v is not None and np.isfinite(v)for v in raw)else np.nan)
            return values
        for rank in ranks:
            for ax,key in zip(axes.flat,("time","relative_output_l2","max_relative_gradient_l2","memory")):
                ax.plot(counts,worst(rank,key),"o-",color=colors.get(rank,"#687881"),label=str(rank)+" features")
        titles=("Worst forward + backward time","Deviation from exact softmax output","Worst shared-parameter gradient deviation","Worst allocated CUDA memory")
        labels=("Fixed kernel / SDPA time ratio","Relative L2 output error","Relative L2 gradient error","Fixed kernel / SDPA memory ratio")
        for i,(ax,title,label)in enumerate(zip(axes.flat,titles,labels)):
            ax.set_title(title);ax.set_ylabel(label);ax.set_xlabel("Tokens N");ax.set_xscale("log",base=2);ax.set_xticks(counts,[str(n)for n in counts]);ax.grid(alpha=.2);ax.legend(fontsize=9)
            if i in(0,3):ax.axhline(1,color="#555555",linestyle="--");ax.set_ylim(bottom=0)
            else:ax.set_ylim(bottom=0)
        fig.suptitle(prefix+"Frozen Notebook15 ViT TRAIN-token audit; gaps remain UNKNOWN")
        save(fig,"kernel_audit.png")
    if cost and cost.get("results"):
        values=cost["results"];fig,axes=plt.subplots(1,2,figsize=(11,4),layout="constrained")
        for i,name in enumerate(("sdpa","fixed")):
            rec=values[name]
            axes[0].bar(i,rec["median_complete_step_seconds"],color=colors[name])
            memory=rec.get("peak_cuda_allocated_bytes")
            if memory is not None:axes[1].bar(i,memory/1024**3,color=colors[name])
        for ax in axes:ax.set_xticks([0,1],["Optimized ViT","Fixed kernel"]);ax.set_ylim(bottom=0);ax.grid(axis="y",alpha=.2)
        axes[0].set_ylabel("Median complete JEPA step (seconds)");axes[1].set_ylabel("Peak allocated CUDA memory (GiB)")
        fig.suptitle(prefix+"Complete JEPA: "+str(cost["image_size"])+" px, equal batch and common input pipeline")
        save(fig,"complete_jepa_cost.png")
    quality=result if result.get("paired_results")else(screen or {})
    if quality.get("paired_results"):
        pairs=quality["paired_results"];fig,axes=plt.subplots(1,2,figsize=(12,5),layout="constrained")
        for j,key,name in[(0,"vit_percent","sdpa"),(1,"candidate_percent","fixed")]:
            values=[r[key]for r in pairs];axes[0].scatter([j]*len(values),values,facecolors="none",edgecolors=colors[name]);axes[0].scatter([j],[np.mean(values)],marker="D",s=70,color=colors[name])
        axes[0].set_xticks([0,1],["ViT","Fixed kernel"]);axes[0].set_ylim(0,max(1,max(r[k]for r in pairs for k in("vit_percent","candidate_percent")))*1.08);axes[0].set_ylabel("Top-1 accuracy (%)")
        delta=[r["delta_pp"]for r in pairs];axes[1].scatter(range(1,len(pairs)+1),delta,color=colors["fixed"]);axes[1].axhline(0,color="#555555");axes[1].axhline(quality["mean_delta_pp"],color=colors["fixed"],linestyle=":",label="Mean paired effect")
        ci=quality.get("raw_ci95_pp")
        if ci is not None:axes[1].axhspan(ci[0],ci[1],color=colors["fixed"],alpha=.15,label="95% paired-t interval for mean")
        else:axes[1].text(.02,.98,"Paired-t interval undefined",transform=axes[1].transAxes,va="top")
        axes[1].set_xticks(list(range(1,len(pairs)+1)));axes[1].set_xlabel("Independent paired training seed index");axes[1].set_ylabel("Candidate − ViT (percentage points)");axes[1].legend(fontsize=9)
        role="Frozen official-subset confirmation"if result.get("official_test","").startswith("EVALUATED")else"Reused-development screening; not confirmation"
        fig.suptitle(prefix+role+" — "+str(len(pairs))+" fresh paired seeds")
        save(fig,"paired_quality.png")
    body=['<!doctype html><html><head><meta charset="utf-8"><title>Notebook17 fixed kernel experiment</title><style>body{font:16px system-ui;color:#17232d;max-width:1100px;margin:40px auto;padding:0 20px}img{width:100%;height:auto}table{border-collapse:collapse;width:100%}th,td{text-align:left;padding:8px;border-bottom:1px solid #dbe2e8}pre{background:#edf2f5;overflow:auto;padding:16px}details{margin:20px 0}</style></head><body>',
          '<h1>Fixed kernel feasibility and JEPA learning</h1><p><strong>'+html.escape(result.get('status','UNKNOWN'))+'</strong> — '+html.escape(str(result.get('reason',result.get('decision','See measured gate receipts'))))+'</p>',
          '<p>'+('SOFTWARE VALIDATION ONLY: these fixtures do not establish accuracy or efficiency.'if fixture else'Actual measured evidence; missing and timed-out cases remain UNKNOWN.')+'</p>',
          '<p>The candidate uses fixed positive kernel features and exact local replacements. This approximates softmax for distant pairs; it is not classical multipole expansion and does not guarantee a superior representation. Independent dense finite-feature checks validate the implementation; softmax approximation errors are reported separately.</p>',
          '<p>Official test: '+html.escape(str(result.get('official_test','CLOSED')))+'. A registered seal is required before predictive access. Input improvements apply equally to both models. Operator microbenchmarks are distinct from complete JEPA steps.</p>']
    if quality.get('paired_results'):
        ci=quality.get('raw_ci95_pp');ci_text='undefined'if ci is None else f'[{ci[0]:+.3f}, {ci[1]:+.3f}] pp'
        body.append('<p>Mean paired difference: '+format(quality['mean_delta_pp'],'+.3f')+' pp; 95% paired-t interval '+ci_text+'. Training seeds are the inferential units; images are not independent training replicates.</p>')
    for name in figures:body.append('<img alt="'+html.escape(name.replace('_',' ').replace('.png',''))+'" src="data:image/png;base64,'+base64.b64encode((artifact/name).read_bytes()).decode()+'">')
    if bench:
        body.append('<h2>Every operator case</h2><table><tr><th>Tokens</th><th>Layer</th><th>Features</th><th>Status</th><th>Output error</th><th>Worst gradient error</th><th>Speed / SDPA</th></tr>')
        for row in bench.get('rows',[]):
            complete=row.get('status')=='MEASURED';error=row.get('errors',{})
            vals=[row.get('tokens'),row.get('layer'),row.get('rank'),row.get('status','UNKNOWN'),format(error['relative_output_l2'],'.4g')if complete else'UNKNOWN',format(error['max_relative_gradient_l2'],'.4g')if complete else'UNKNOWN',format(row['operator_speed_over_sdpa'],'.4g')if complete else'UNKNOWN']
            body.append('<tr>'+''.join('<td>'+html.escape(str(v))+'</td>'for v in vals)+'</tr>')
        body.append('</table>')
    for name,value in(('Shared input pipeline',pipeline),('Kernel audit',bench),('Complete JEPA cost',cost),('Development screening',screen),('Complete result',result)):
        if value is not None:body.append('<details><summary>'+name+'</summary><pre>'+html.escape(json.dumps(value,indent=2,default=str))+'</pre></details>')
    body.append('<p>Source evidence: result.json, case/source/checkpoint manifests and phase clocks retained beside this report.</p></body></html>')
    (artifact/'report.html').write_text('\n'.join(body))
    target=artifact/'result.json';temporary=target.with_name(target.name+'.'+uuid.uuid4().hex+'.tmp')
    with temporary.open('w')as f:
        f.write(json.dumps(result,indent=2,allow_nan=False,default=str)+'\n');f.flush();os.fsync(f.fileno())
    os.replace(temporary,target)
    return {'html':str(artifact/'report.html'),'figures':figures,'result':str(artifact/'result.json')}
